In [1]:
!nvidia-smi

Sun Apr  5 18:11:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             33W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install accelerate
!pip install --upgrade transformers
!pip install bitsandbytes
!pip install datasets
!pip install rouge-score
!pip install pymorphy2
!pip install peft
#!pip install unsloth
#!pip install flash_attn

In [3]:
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

class ChatDataset(Dataset):
    def __init__(self, records: list, tokenizer, max_length: int = 1024, ignore_index: int = -100):
        self.records = []

        for record in tqdm(records, desc="Подготовка датасета"):
            input_ids = []
            labels = []

            for message in record["messages"]:
                # в saiga датасете старый формат ролей "bot"
                role = "assistant" if message["role"] in ["assistant", "bot", "gpt"] else message["role"]
                tokens = tokenizer.apply_chat_template(
                    [{"role": role, "content": message["content"]}],
                    tokenize=True,
                    add_generation_prompt=False,
                    add_special_tokens=False
                )['input_ids']

                if len(input_ids) + len(tokens) > max_length:
                    break

                # учимся только на ответах ассистента, не на вопросах
                if role == "assistant":
                    step_labels = tokens
                else:
                    step_labels = [ignore_index] * len(tokens)

                input_ids.extend(tokens)
                labels.extend(step_labels)

            if not input_ids:
                continue

            # Проверка, что в примере не только вопрос, но есть хотя бы 1 ответ
            has_target_tokens = any(label != ignore_index for label in labels)
            if not has_target_tokens:
                continue

            # Конвертируем в torch тензоры
            self.records.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long)
            })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]

In [4]:
import random
import json
import os

import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForTokenClassification,
    AutoConfig,
)
from transformers import (
    Trainer,
    TrainingArguments,
    logging,
    TrainerCallback,
    TrainerState,
    TrainerControl,
    BitsAndBytesConfig,
)
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR
from peft import get_peft_model, LoraConfig
import re
from peft import prepare_model_for_kbit_training
import codecs

os.environ["WANDB_DISABLED"] = "true"

In [5]:
model_name = 'LiquidAI/LFM2.5-1.2B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=f"cuda:0",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [7]:
model

Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Li

In [8]:
from datasets import load_dataset

In [12]:
coat_full = load_dataset("RussianNLP/coat", "binary", trust_remote_code=True)['train']

coat_full = coat_full.shuffle(seed=42)

coat_test_raw = coat_full.select(range(100))
coat_trainval_raw = coat_full.select(range(100, 2500))

n_trainval = len(coat_trainval_raw)
n_train = int(0.9 * n_trainval)
coat_train_raw = coat_trainval_raw.select(range(n_train))
coat_val_raw = coat_trainval_raw.select(range(n_train, n_trainval))

print(f"COAT train: {len(coat_train_raw)}, val: {len(coat_val_raw)}, test: {len(coat_test_raw)}")

def make_coat_dialogs(data):
    dialogs = []
    for ex in data:
        text = ex['text']
        label = ex['label']  # 0=человек, 1=ии
        answer = "человек" if label == 0 else "ии"
        prompt = f"Определи, является ли следующий текст написанным человеком или сгенерированным искусственным интеллектом. Ответь только одним словом: 'человек' или 'ии'.\n\nТекст: {text}\n\nОтвет:"
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer}
        ]
        dialogs.append({"messages": messages})
    return dialogs

coat_train_dialogs = make_coat_dialogs(coat_train_raw)
coat_val_dialogs = make_coat_dialogs(coat_val_raw)
coat_test_dialogs = make_coat_dialogs(coat_test_raw)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'RussianNLP/coat' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'RussianNLP/coat' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


COAT train: 2160, val: 240, test: 100


In [13]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import re
import numpy as np
from collections import Counter
import requests
import zipfile
from datasets import concatenate_datasets

In [15]:
import requests, zipfile
url = "https://russiansuperglue.com/tasks/download/PARus"
response = requests.get(url)
with open("PARus.zip", "wb") as f:
    f.write(response.content)
with zipfile.ZipFile("PARus.zip", "r") as zip_ref:
    zip_ref.extractall("PARus_data")

parus_train = load_dataset('json', data_files="PARus_data/PARus/train.jsonl")['train']
parus_val = load_dataset('json', data_files="PARus_data/PARus/val.jsonl")['train']

parus_all_labeled = concatenate_datasets([parus_train, parus_val])
parus_all_labeled = parus_all_labeled.shuffle(seed=42)

parus_test_raw = parus_all_labeled.select(range(100))
parus_trainval_raw = parus_all_labeled.select(range(100, len(parus_all_labeled)))

n_trainval = len(parus_trainval_raw)
n_train = int(0.9 * n_trainval)
parus_train_raw = parus_trainval_raw.select(range(n_train))
parus_val_raw = parus_trainval_raw.select(range(n_train, n_trainval))

print(f"PARus train: {len(parus_train_raw)}, val: {len(parus_val_raw)}, test: {len(parus_test_raw)}")

def make_parus_dialogs(data):
    dialogs = []
    for ex in data:
        premise = ex['premise']
        choice1 = ex['choice1']
        choice2 = ex['choice2']
        question = ex['question']
        label = ex['label']
        q_text = "причиной" if question == "cause" else "следствием"
        prompt = (f"Определи, какой из двух вариантов является более правдоподобным {q_text} для следующего утверждения.\n"
                  f"Утверждение: {premise}\n"
                  f"Вариант 1: {choice1}\n"
                  f"Вариант 2: {choice2}\n"
                  f"Ответь только номером варианта: 1 или 2.")
        answer = "1" if label == 0 else "2"
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer}
        ]
        dialogs.append({"messages": messages})
    return dialogs

parus_train_dialogs = make_parus_dialogs(parus_train_raw)
parus_val_dialogs = make_parus_dialogs(parus_val_raw)
parus_test_dialogs = make_parus_dialogs(parus_test_raw)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

PARus train: 360, val: 40, test: 100


In [16]:
train_dialogs = coat_train_dialogs + parus_train_dialogs
val_dialogs = coat_val_dialogs + parus_val_dialogs
test_dialogs = coat_test_dialogs + parus_test_dialogs

print(f"Train диалогов: {len(train_dialogs)}")
print(f"Val диалогов:   {len(val_dialogs)}")
print(f"Test диалогов:  {len(test_dialogs)}")

Train диалогов: 2520
Val диалогов:   280
Test диалогов:  200


In [17]:
max_length = 1024
train_dataset = ChatDataset(train_dialogs, tokenizer, max_length=max_length)
val_dataset = ChatDataset(val_dialogs, tokenizer, max_length=max_length)
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Подготовка датасета:   0%|          | 0/2520 [00:00<?, ?it/s]

Подготовка датасета:   0%|          | 0/280 [00:00<?, ?it/s]

Train samples: 2518, Val samples: 280


In [18]:
training_args = {
    "output_dir": "./lfm_instruct",
    "eval_strategy": "steps",
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "eval_steps": 16,
    "logging_steps": 1,
    "learning_rate": 2e-5,
    "num_train_epochs": 2,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 16,
    "bf16": False,
    "fp16": True,
    "optim": "adamw_torch",
    "save_strategy": "no",
    "load_best_model_at_end": False,
    "gradient_checkpointing": True,
    "seed": 1337,
    "max_grad_norm": 1.0,
    "weight_decay": 0.05
}
training_args = TrainingArguments(**training_args)

In [19]:
# замораживаем все веса
for param in model.parameters():
    param.requires_grad = False

# размораживаем эмбеддинги
model.get_input_embeddings().requires_grad_(True)
model.get_output_embeddings().requires_grad_(True)

# размораживаем последние 2 слоя
total_layers = len(model.model.layers)
for i in range(total_layers - 2, total_layers):
    for param in model.model.layers[i].parameters():
        param.requires_grad = True
if hasattr(model.model, 'embedding_norm'):
    model.model.embedding_norm.weight.requires_grad = True

# для стабильности обучения, чтобы не было nan, обучаемые мастер веса в fp32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable / 1e6:.1f}M / {total / 1e6:.1f}M "
      f"({100 * trainable / total:.1f}%)")

Trainable: 262.2M / 1170.3M (22.4%)


In [20]:
data_collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

if len(trainer.label_names) == 0:
    trainer.label_names.append('labels')

In [21]:
trainer.train()

Step,Training Loss,Validation Loss
16,1.721281,1.904894
32,0.121524,0.146109
48,0.087064,0.092599
64,0.153153,0.096059
80,0.123169,0.089713
96,0.075745,0.083065
112,0.099309,0.085299
128,0.077034,0.091763
144,0.088453,0.090880
160,0.076840,0.091723


TrainOutput(global_step=630, training_loss=0.21376056421015943, metrics={'train_runtime': 1393.9096, 'train_samples_per_second': 3.613, 'train_steps_per_second': 0.452, 'total_flos': 5110555915345920.0, 'train_loss': 0.21376056421015943, 'epoch': 2.0})

In [22]:
model.save_pretrained('./lfm_instruct')
tokenizer.save_pretrained('./lfm_instruct')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./lfm_instruct/tokenizer_config.json',
 './lfm_instruct/chat_template.jinja',
 './lfm_instruct/tokenizer.json')

In [24]:
model.eval()

Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Li

In [25]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(model_name)
generation_config

GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 7,
  "pad_token_id": 0
}

In [26]:
generation_config.temperature = 0.3
generation_config.max_new_tokens = 512
generation_config

GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 7,
  "max_new_tokens": 512,
  "pad_token_id": 0,
  "temperature": 0.3
}

In [29]:
test_sample = coat_test_raw

def build_prompt(text):
    messages = [
        {"role": "user", "content": f"Определи, является ли следующий текст написанным человеком или сгенерированным искусственным интеллектом. Ответь только одним словом: 'человек' или 'ии'.\n\nТекст: {text}\n\nОтвет:"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

def get_model_prediction(text):
    prompt = build_prompt(text)
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_ids = outputs[0][input_ids.shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()
    return answer

def extract_label(answer):
    if 'человек' in answer or 'human' in answer:
        return 0
    elif 'ии' in answer or 'ai' in answer or 'machine' in answer or 'generated' in answer:
        return 1
    else:
        return None

true_labels = []
pred_labels = []
texts = []

for example in test_sample:
    text = example['text']
    true_label = example['label']
    texts.append(text)
    true_labels.append(true_label)

    answer = get_model_prediction(text)
    pred = extract_label(answer)
    pred_labels.append(pred if pred is not None else -1)

    if len(true_labels) <= 5:
        print(f"Текст: {text[:100]}...")
        print(f"Ответ модели: {answer}")
        print(f"Истинная метка: {true_label}, предсказанная: {pred}")
        print("-" * 50)

valid_indices = [i for i, p in enumerate(pred_labels) if p != -1]
if len(valid_indices) < len(pred_labels):
    print(f"Не удалось распознать ответ для {len(pred_labels) - len(valid_indices)} примеров. Они исключены из оценки.")
    true_labels = [true_labels[i] for i in valid_indices]
    pred_labels = [pred_labels[i] for i in valid_indices]

accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')

print(f"\nРезультаты zero-shot оценки на {len(true_labels)} примерах:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Текст: Вначале, роль АВС была ограничена вопросами стандартизации military equipment, training, и тактики....
Ответ модели: ии
Истинная метка: 1, предсказанная: 1
--------------------------------------------------
Текст: Он участвовал в телеконференции с президентом РФ, которая состоялась накануне. По мнению президента ...
Ответ модели: ии
Истинная метка: 1, предсказанная: 1
--------------------------------------------------
Текст: Комплексная организация отдыха и развлечений в городе Туапсе. Отдых на море, отдых по доступным цена...
Ответ модели: ии
Истинная метка: 1, предсказанная: 1
--------------------------------------------------
Текст: Органы местного самоуправления обязаны содержать, обслуживать и обеспечивать сохранность временно не...
Ответ модели: человек
Истинная метка: 0, предсказанная: 0
--------------------------------------------------
Текст: Эти тексты описывают происхождение и природу Восточной Чистой Земли, в которой находится Будда Амита...
Ответ модели: ии
Истинная

In [30]:
train_sample = parus_test_raw

def build_parus_prompt(premise, choice1, choice2, question):
    if question == "cause":
        q_text = "причиной"
    else:
        q_text = "следствием"
    prompt = (
        f"Определи, какой из двух вариантов является более правдоподобным {q_text} для следующего утверждения.\n"
        f"Утверждение: {premise}\n"
        f"Вариант 1: {choice1}\n"
        f"Вариант 2: {choice2}\n"
        f"Ответь только номером варианта: 1 или 2."
    )
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def get_prediction(premise, choice1, choice2, question):
    prompt = build_parus_prompt(premise, choice1, choice2, question)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    if '1' in answer:
        return 0
    elif '2' in answer:
        return 1
    else:
        return None

true_labels = []
pred_labels = []

for example in train_sample:
    premise = example['premise']
    choice1 = example['choice1']
    choice2 = example['choice2']
    question = example['question']
    true_label = example['label']

    pred = get_prediction(premise, choice1, choice2, question)
    true_labels.append(true_label)
    pred_labels.append(pred)

    if len(true_labels) <= 5:
        print(f"Premise: {premise[:60]}...")
        print(f"Q: {question}, Choice1: {choice1[:30]}..., Choice2: {choice2[:30]}...")
        print(f"True: {true_label} -> {'choice1' if true_label==0 else 'choice2'}, Pred: {pred}")
        print("-" * 40)

valid = [i for i, p in enumerate(pred_labels) if p is not None]
true_labels = [true_labels[i] for i in valid]
pred_labels = [pred_labels[i] for i in valid]

if true_labels:
    acc = accuracy_score(true_labels, pred_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')
    print(f"\n✅ Accuracy на {len(true_labels)} примерах: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
else:
    print("❌ Нет валидных предсказаний.")

Premise: Лето сменилось осенью....
Q: effect, Choice1: Людей эвакуировали из домов...., Choice2: Листья упали с деревьев....
True: 1 -> choice2, Pred: 1
----------------------------------------
Premise: Группа закрыла глаза на бестактность, допущенную женщиной....
Q: effect, Choice1: Женщина была прощена...., Choice2: Женщина извинилась....
True: 0 -> choice1, Pred: 0
----------------------------------------
Premise: Учитель задал студентам домашнее задание....
Q: effect, Choice1: Студенты сдали записи...., Choice2: Студенты застонали....
True: 1 -> choice2, Pred: 1
----------------------------------------
Premise: Женщина дрожала, выйдя из бассейна....
Q: effect, Choice1: Она завернулась в полотенце...., Choice2: Она налила себе лимонада....
True: 0 -> choice1, Pred: 0
----------------------------------------
Premise: Жилец потерял ключи от квартиры....
Q: effect, Choice1: Хозяин квартиры открыл ему две..., Choice2: Хозяин квартиры починил дверь....
True: 0 -> choice1, Pred: 1
-------